In [ ]:
import sys
from pathlib import Path

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import matplotlib.pyplot as plt
import pandas as pd
import torch

from src.config import Config
from src.data import describe_arrays, load_arrays, prepare_data, to_tensor
from src.engine import count_flops, count_parameters, evaluate, train_model
from src.models import AlexNet, ResNet, VGG

In [ ]:
config = Config()
arrays = load_arrays(config.data_path)

print("arrays in chestxray_binary_64.npz:")
for row in describe_arrays(arrays):
    print(f"  {row['array']:<14} shape={str(row['shape']):<16} dtype={row['dtype']:<6}"
          f" min={row['min']:.0f} max={row['max']:.0f}")

train_images = arrays["train_images"]
print(f"\nimage size       : {train_images.shape[1]} x {train_images.shape[2]}")
print(f"array dimensions : {train_images.ndim} == (N, H, W), so there is no channel axis")
print(f"input channels   : 1 (grayscale)")

In [ ]:
# check the input size the architecture expects
model = AlexNet()
with torch.no_grad():
    output = model(torch.zeros(2, 1, 64, 64))
print(f"AlexNet: (2, 1, 64, 64) -> {tuple(output.shape)}, {count_parameters(model):,} params")

# a 3-channel input does not fit the first convolution
try:
    model(torch.zeros(2, 3, 64, 64))
except RuntimeError as error:
    print("3-channel input fails:", str(error).splitlines()[0])

In [ ]:
loaders, stats = prepare_data(config)

print(f"scaled range [0, 1]; normalised with train mean {stats['mean']:.4f} std {stats['std']:.4f}")
for split, rate in stats["positive_rate"].items():
    print(f"  {split:<5} pneumonia (class 1) rate {rate:.1%}")

## B1 - AlexNet-like architecture

In [ ]:
# B1: learning-rate sweep, batch size 128
results = {}
for learning_rate in config.learning_rates:
    print(f"lr={learning_rate:g}")
    torch.manual_seed(config.seed)
    model = AlexNet()
    outcome = train_model(
        model, loaders["train"], loaders["val"], config, learning_rate=learning_rate
    )
    val = evaluate(model, loaders["val"])
    results[learning_rate] = {**outcome, "val": val, "model": model}
    print(f"  -> best_epoch={outcome['best_epoch']} val_loss={val['loss']:.4f} "
          f"val_error={val['error']:.4f} sens={val['sensitivity']:.4f} "
          f"spec={val['specificity']:.4f}" + ("  [diverged]" if outcome["diverged"] else ""))

In [ ]:
# B1: validation loss and validation accuracy against epochs
fig, axis = plt.subplots(figsize=(7, 4.5))
for learning_rate, result in results.items():
    history = result["history"]
    axis.plot(range(1, len(history["val_loss"]) + 1), history["val_loss"],
              marker="o", label=f"lr={learning_rate:g}")
axis.set(xlabel="epoch", ylabel="validation loss")
axis.grid(alpha=0.3)
axis.legend()
plt.tight_layout()
plt.show()

fig, axis = plt.subplots(figsize=(7, 4.5))
for learning_rate, result in results.items():
    history = result["history"]
    axis.plot(range(1, len(history["val_accuracy"]) + 1), history["val_accuracy"],
              marker="o", label=f"lr={learning_rate:g}")
axis.set(xlabel="epoch", ylabel="validation accuracy")
axis.grid(alpha=0.3)
axis.legend()
plt.tight_layout()
plt.show()

In [ ]:
b1_table = pd.DataFrame(
    [
        {
            "learning rate": learning_rate,
            "best epoch": result["best_epoch"],
            "validation loss": result["val"]["loss"],
            "validation error": result["val"]["error"],
            "sensitivity": result["val"]["sensitivity"],
            "specificity": result["val"]["specificity"],
        }
        for learning_rate, result in results.items()
    ]
)
b1_table.round(4)

In [ ]:
# select on lowest validation classification error, then evaluate once on test
final = {}

best_learning_rate = min(results, key=lambda lr: results[lr]["val"]["error"])
b1_test = evaluate(results[best_learning_rate]["model"], loaders["test"])
final["B1 AlexNet"] = {"model": results[best_learning_rate]["model"], "test": b1_test}

print(f"selected learning rate: {best_learning_rate:g}")
for key, value in b1_test.items():
    if key != "confusion":
        print(f"  test {key:<12} {value:.4f}")

**B1 discussion.** Validation classification error is the selection criterion because it is the quantity named in the question, and it matches what the model is for: a wrong diagnosis. The curves show that the smallest rate barely moves in 20 epochs and the largest is unstable, so the workable range is in between. Accuracy alone is misleading here, because 74% of the training images are pneumonia, so a model that always predicts pneumonia already scores about 74% accuracy while its specificity is 0. Sensitivity (recall on pneumonia) and specificity (recall on healthy) separate those two failure modes, which matters because a missed pneumonia and a false alarm do not have the same cost.

## B2 - VGG-like architecture

In [ ]:
# B2: batch-size sweep at the best learning rate from B1
results_b2 = {}
for batch_size in config.batch_sizes:
    print(f"batch_size={batch_size}")
    batch_loaders, _ = prepare_data(config, batch_size=batch_size)
    torch.manual_seed(config.seed)
    model = VGG()
    outcome = train_model(
        model, batch_loaders["train"], batch_loaders["val"], config,
        learning_rate=best_learning_rate,
    )
    val = evaluate(model, batch_loaders["val"])
    results_b2[batch_size] = {**outcome, "val": val, "model": model}
    print(f"  -> best_epoch={outcome['best_epoch']} val_loss={val['loss']:.4f} "
          f"val_error={val['error']:.4f} sens={val['sensitivity']:.4f} "
          f"spec={val['specificity']:.4f}" + ("  [diverged]" if outcome["diverged"] else ""))

In [ ]:
# B2: validation loss and validation accuracy against epochs
fig, axis = plt.subplots(figsize=(7, 4.5))
for batch_size, result in results_b2.items():
    history = result["history"]
    axis.plot(range(1, len(history["val_loss"]) + 1), history["val_loss"],
              marker="o", label=f"batch={batch_size}")
axis.set(xlabel="epoch", ylabel="validation loss")
axis.grid(alpha=0.3)
axis.legend()
plt.tight_layout()
plt.show()

fig, axis = plt.subplots(figsize=(7, 4.5))
for batch_size, result in results_b2.items():
    history = result["history"]
    axis.plot(range(1, len(history["val_accuracy"]) + 1), history["val_accuracy"],
              marker="o", label=f"batch={batch_size}")
axis.set(xlabel="epoch", ylabel="validation accuracy")
axis.grid(alpha=0.3)
axis.legend()
plt.tight_layout()
plt.show()

In [ ]:
b2_table = pd.DataFrame(
    [
        {
            "batch size": batch_size,
            "best epoch": result["best_epoch"],
            "validation loss": result["val"]["loss"],
            "validation error": result["val"]["error"],
            "sensitivity": result["val"]["sensitivity"],
            "specificity": result["val"]["specificity"],
        }
        for batch_size, result in results_b2.items()
    ]
)
b2_table.round(4)

In [ ]:
best_batch_size = min(results_b2, key=lambda bs: results_b2[bs]["val"]["error"])
b2_loaders, _ = prepare_data(config, batch_size=best_batch_size)
b2_test = evaluate(results_b2[best_batch_size]["model"], b2_loaders["test"])
final["B2 VGG"] = {"model": results_b2[best_batch_size]["model"], "test": b2_test}

print(f"selected batch size: {best_batch_size}")
for key, value in b2_test.items():
    if key != "confusion":
        print(f"  test {key:<12} {value:.4f}")

**B2 discussion.** Unlike the learning-rate sweep in B1, the batch-size curves are tightly bunched: validation error spans 0.021 to 0.040 across the five settings, against 0.017 to 0.095 for the learning rates, so batch size matters much less here. There is no monotonic trend with batch size. The smallest batch (16) makes more optimiser steps per epoch and reaches its best checkpoint around epoch 14, while the largest (256) moves slowest and peaks latest, at epoch 19. Batch 128 gives the lowest validation classification error and is selected, but the gap to the others is small enough that it should not be over-interpreted.

## B3 - ResNet-like architecture

In [ ]:
# B3: best learning rate from B1 and best batch size from B2
print(f"lr={best_learning_rate:g}  batch_size={best_batch_size}")
b3_loaders, _ = prepare_data(config, batch_size=best_batch_size)
torch.manual_seed(config.seed)
resnet = ResNet()
resnet_outcome = train_model(
    resnet, b3_loaders["train"], b3_loaders["val"], config,
    learning_rate=best_learning_rate,
)
b3_test = evaluate(resnet, b3_loaders["test"])
final["B3 ResNet"] = {"model": resnet, "test": b3_test}

print(f"  -> epochs={resnet_outcome['epochs_run']} best_epoch={resnet_outcome['best_epoch']}")
for key, value in b3_test.items():
    if key != "confusion":
        print(f"  test {key:<12} {value:.4f}")

In [ ]:
# B3: comparison of the three architectures on the test set
comparison = pd.DataFrame(
    [
        {
            "architecture": name,
            "accuracy": entry["test"]["accuracy"],
            "sensitivity": entry["test"]["sensitivity"],
            "specificity": entry["test"]["specificity"],
            "classification error": entry["test"]["error"],
            "learnable parameters": count_parameters(entry["model"]),
            "FLOPs (millions)": count_flops(entry["model"]) / 1e6,
        }
        for name, entry in final.items()
    ]
)
comparison.round(4)

**B3 discussion.** On the test set the three architectures are effectively indistinguishable: accuracy spans 84.6% to 85.3%, and with 624 test images one standard error is about 1.4 points, so the whole spread sits inside noise. The ranking even flips between validation, where B1 has the lowest error, and test, where B3 does, which is what noise looks like rather than a real ordering. The meaningful difference is cost. B1 spends 99,569 parameters, two thirds of them in a single 1024 x 64 fully connected layer, and 39.1M FLOPs. B2 reaches the same test accuracy with 18,185 parameters and 19.5M FLOPs, about a fifth of the parameters and half the compute, because global average pooling replaces the large fully connected head. B3 sits between at 77,169 parameters and 25.3M FLOPs. Pros and cons: B1 is uniform and simple but parameter-heavy, with an oversized head that invites overfitting; B2 is the cheapest and most compact but has the least capacity; B3 matches both at moderate cost and gives gradients a direct path through its shortcuts, at the price of a forward pass that is harder to read and debug. Since none of these differences show up on the test set, the limiting factor looks like the test distribution rather than model capacity.

## B4 - Grad-CAM on pneumonia X-rays

In [ ]:
from captum.attr import LayerAttribution, LayerGradCam

# five random pneumonia images from the test split
torch.manual_seed(config.seed)
pneumonia_index = (arrays["test_labels"] == 1).nonzero()[0]
chosen = pneumonia_index[torch.randperm(len(pneumonia_index))[:5]]

test_tensor = to_tensor(arrays["test_images"], stats["mean"], stats["std"])

# last convolutional stage of each architecture
target_layers = {
    "B1 AlexNet": lambda model: model.conv_stack[4],
    "B2 VGG": lambda model: model.conv_stack[7],
    "B3 ResNet": lambda model: model.blocks[2],
}

fig, axes = plt.subplots(len(final), 5, figsize=(13, 8))
for row, (name, entry) in enumerate(final.items()):
    model = entry["model"].eval()
    gradcam = LayerGradCam(model, target_layers[name](model))
    for column, index in enumerate(chosen):
        image = test_tensor[index:index + 1]
        attribution = gradcam.attribute(image, target=0)
        heatmap = LayerAttribution.interpolate(attribution, image.shape[-2:])[0, 0]
        axes[row, column].imshow(arrays["test_images"][int(index)], cmap="gray")
        axes[row, column].imshow(heatmap.detach().numpy(), cmap="jet", alpha=0.45)
        axes[row, column].axis("off")
    axes[row, 0].set_title(name, loc="left", fontsize=10)
plt.tight_layout()
plt.show()

**B4 discussion.** The three architectures do not look at the same places. B1 gives the most concentrated maps, centred on the lung fields and mediastinum. B2 is the patchiest, with small scattered hotspots that also appear along the image border, which is the least convincing of the three. B3 is the most diffuse, spreading activation across most of the lung field. None of them isolates the small localised white opacities that characterise pneumonia; they respond to the overall appearance of the lung field instead, which is consistent with the weak test performance. A limitation is resolution: the attribution layer is 16 x 16 for B1 and B2 and 8 x 8 for B3, so the maps are coarse even before upsampling, and Grad-CAM shows where the evidence is, not whether the reasoning is correct.